# Stabilité du langage des filings — la jambe longue de *Lazy Prices*

**Référence :** qc-research #20966 « Filing language stability as a selection signal » (Emily Xinyu Sun, Triton Quantitative Trading @ UCSD) ; projet compagnon `main.py` (backtest QC Cloud).

**Cadre :** Cohen, Malloy & Nguyen, *Lazy Prices*, Journal of Finance 75(3), 2020, DOI 10.1111/jofi.12885.

## Idée

*Lazy Prices* montre que le **changement** de langage des sections à risque (Item 1A) des 10-K prédit les mauvaises nouvelles : les entreprises qui **réécrivent** abondamment leurs filings sur-performent négativement, les « non-changers » (filings quasi identiques d'une année sur l'autre) sur-performent positivement. L'article #20966 n'implémente que la **jambe longue** : acheter les filers les plus *stables*, classés par similarité de langage (dataset **Brain Language Metrics on Company Filings** sur QuantConnect), pondérés max-Sharpe.

Ce notebook construit un **vérificateur local indépendant** du concept : on télécharge les vraies sections Item 1A depuis **SEC EDGAR**, on calcule une similarité TF-IDF cosinus entre années consécutives (même famille que la métrique Brain), et on range un petit panier par stabilité.

## 1. Panier et téléchargement EDGAR

Données réelles : API `data.sec.gov/submissions` (métadonnées de filings) puis le document primaire de chaque 10-K. Le SEC impose un `User-Agent` identifiant et un rythme modéré (max ~10 requêtes/s ; nous restons très en dessous).

Panier : 5 grandes capitalisations aux filings réguliers, **deux 10-K consécutifs** chacune (l'année N et N-1 : la similarité se mesure entre voisins).

In [1]:
import json
import re
import time
import html as ihtml
import urllib.request

import pandas as pd

TICKERS = {
    "AAPL": 320193,
    "MSFT": 789019,
    "KO": 21344,
    "WMT": 104169,
    "GE": 40545,
}

UA = {"User-Agent": "CoursIA Research jsboige@gmail.com"}


def http_get(url):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=60) as resp:
        return resp.read()


def latest_10k_accessions(cik, n=2):
    data = json.loads(http_get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json"))
    recent = data["filings"]["recent"]
    pairs = []
    for form, acc, doc, filed in zip(
            recent["form"], recent["accessionNumber"],
            recent["primaryDocument"], recent["filingDate"]):
        if form == "10-K":
            pairs.append((acc, doc, filed))
        if len(pairs) == n:
            break
    return pairs


print("Filtrage des 10-K les plus récents par ticker :")
edgar = {}
for ticker, cik in TICKERS.items():
    accs = latest_10k_accessions(cik)
    edgar[ticker] = accs
    time.sleep(0.4)  # courtoisie EDGAR
    print(f"  {ticker}: " + ", ".join(f"{filed} ({acc})" for acc, _, filed in accs))

Filtrage des 10-K les plus récents par ticker :


  AAPL: 2025-10-31 (0000320193-25-000079), 2024-11-01 (0000320193-24-000123)


  MSFT: 2026-07-29 (0001193125-26-323660), 2025-07-30 (0000950170-25-100235)


  KO: 2026-02-20 (0001628280-26-010047), 2025-02-20 (0000021344-25-000011)


  WMT: 2026-03-13 (0000104169-26-000055), 2025-03-14 (0000104169-25-000021)


  GE: 2026-01-29 (0000040545-26-000008), 2025-02-03 (0000040545-25-000015)


Chaque 10-K est un document HTML volumineux (1 à 8 Mo). On en extrait la section **Item 1A — Risk Factors** par bornage regex (`Item 1A` → `Item 1B`), avec repli honnête sur le texte intégral tronqué si la borne n'est pas trouvée — le mode retenu est affiché.

In [2]:
TAG = re.compile(r"<[^>]+>")
WS = re.compile(r"\s+")


def filing_text(cik, accession, primary_doc):
    acc_nodash = accession.replace("-", "")
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_nodash}/{primary_doc}"
    raw = http_get(url).decode("utf-8", errors="replace")
    text = WS.sub(" ", TAG.sub(" ", ihtml.unescape(raw)))
    return text


def item_1a(text):
    m = re.search(r"Item\s*1A\b[^a-zA-Z0-9]{0,40}Risk Factors(.{200,}?)Item\s*1B\b",
                  text, re.IGNORECASE | re.DOTALL)
    if m and len(m.group(1)) > 5000:
        return m.group(1), "item_1a"
    return text[:250_000], "full_fallback"


sections = {}
for ticker, accs in edgar.items():
    for acc, doc, filed in accs:
        text = filing_text(TICKERS[ticker], acc, doc)
        sec, mode = item_1a(text)
        sections[(ticker, filed)] = (sec, mode)
        time.sleep(0.4)

modes = pd.Series({f"{t} {f}": m for (t, f), (_, m) in sections.items()})
sizes = pd.Series({f"{t} {f}": len(s) for (t, f), (s, _) in sections.items()})
print("Longueurs de sections extraites (caractères) et mode :")
print(pd.DataFrame({"chars": sizes, "mode": modes}).to_string())

Longueurs de sections extraites (caractères) et mode :
                  chars           mode
AAPL 2025-10-31   87335        item_1a
AAPL 2024-11-01   87724        item_1a
MSFT 2026-07-29  125530        item_1a
MSFT 2025-07-30  120772        item_1a
KO 2026-02-20    150557        item_1a
KO 2025-02-20    148758        item_1a
WMT 2026-03-13   140736        item_1a
WMT 2025-03-14   147248        item_1a
GE 2026-01-29    250000  full_fallback
GE 2025-02-03    250000  full_fallback


## 2. Score de stabilité — TF-IDF cosinus entre années voisines

Même famille que la métrique du fournisseur Brain : une similarité proche de 1 signifie un Item 1A quasi inchangé (filier « paresseux » au sens de *Lazy Prices*), proche de 0 une réécriture massive. On utilise `TfidfVectorizer` (mots, stopwords anglais, 1-2 grammes) puis similarité cosinus.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def stability_score(text_a, text_b):
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                          max_features=60_000, sublinear_tf=True)
    mat = vec.fit_transform([text_a, text_b])
    return float(cosine_similarity(mat[0], mat[1])[0, 0])


rows = []
for ticker in TICKERS:
    pair = sorted((filed for (t, filed) in sections if t == ticker))
    if len(pair) < 2:
        continue
    older, newer = pair[-2], pair[-1]
    text_a, mode_a = sections[(ticker, older)]
    text_b, mode_b = sections[(ticker, newer)]
    score = stability_score(text_a, text_b)
    rows.append({"ticker": ticker, "de": older, "a": newer,
                 "similarite": round(score, 4),
                 "mode": f"{mode_a}/{mode_b}"})

stability = pd.DataFrame(rows).sort_values("similarite", ascending=False).reset_index(drop=True)
stability["rang"] = stability.index + 1
print(stability.to_string(index=False))

ticker         de          a  similarite                        mode  rang
    KO 2025-02-20 2026-02-20      0.9654             item_1a/item_1a     1
  MSFT 2025-07-30 2026-07-29      0.8403             item_1a/item_1a     2
  AAPL 2024-11-01 2025-10-31      0.8346             item_1a/item_1a     3
   WMT 2025-03-14 2026-03-13      0.8065             item_1a/item_1a     4
    GE 2025-02-03 2026-01-29      0.7644 full_fallback/full_fallback     5


## 3. Interprétation — que classe ce score ?

Le ranking ci-dessus est le **micro-équivalent local** de l'univers de la stratégie Cloud : `main.py` prend le top 25 d'un univers de 100 actions liquides classées par cette même famille de similarité (métrique propriétaire Brain sur risk-factors avec repli rapport complet — cf `_select_assets`).

Points de lecture :

- une similarité Item 1A **élevée** = filer stable = candidat long (jambe *Lazy Prices*) ;
- l'article source rapporte Sharpe **0.558** (janv. 2020 - juin 2026) contre **0.533** pour le SPY buy-and-hold — un edge **marginal** (+0.025), et l'auteure attribue le résultat surtout à la **fenêtre de l'optimiseur** max-Sharpe plutôt qu'à la largeur du panier ; seules **9/25** combinaisons du sweep (lookback × taille d'univers) battent le benchmark (36 %) ;
- la **preuve formelle** n'existe pas ici : ce notebook calcule sur données réelles EDGAR le score, le backtest Cloud mesure la performance — ni l'un ni l'autre ne « prouve » l'anomalie ; *Lazy Prices* (JF 2020) porte le poids académique de la proposition.

## 4. Backtest QC Cloud compagnon

Le projet `main.py` (projet Cloud `Filing-Language-Stability`) rejoue le pipeline : filtre liquidité top 100 → similarité Brain top 25 → poids max-Sharpe 12 mois, rebalancement mensuel, janv. 2020 - juin 2026. Les métriques du run sont reportées dans le README du projet (Sharpe / CAGR / MaxDD / PSR) — à lire **avec** le verdict honnête : l'article lui-même conclut à un edge fragile.

## 5. Exercices

> Stubs **sans** `raise NotImplementedError` (règle C.1) : le notebook s'exécute de bout en bout même non complété.

In [4]:
# Exercice 1 -- Votre propre panier
# Indice: ajoutez 2 tickers de votre choix dans TICKERS (CIK sur sec.gov/cgi-bin/browse-edgar),
# relancez les cellules, et observez comment la similarite se deplace.
result = None  # TODO etudiant: noter les nouveaux scores et rangs
print("Exercice a completer")

Exercice a completer


In [5]:
# Exercice 2 -- Sensibilite au preprocessing
# Indice: retirez stop_words ou passez ngram_range=(1,3) dans stability_score.
# Le RANG des tickers change-t-il ? C'est la question de robustesse que le sweep
# 5x5 de l'article (lookback x taille d'univers) pose cote portefeuille.
result = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


In [6]:
# Exercice 3 -- Extraire une annee de plus et mesurer la DERIVE
# Indice: passez n=3 dans latest_10k_accessions, calculez la similarite N-2->N-1
# puis N-1->N ; un filer dont la similarite CHUTE brutalement est un "changer"
# au sens de Lazy Prices -- la jambe courte que la strategie ne prend pas.
result = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


## Conclusion

- **SOTA-OK local** : données réelles SEC EDGAR (sections Item 1A extraites des vrais 10-K), similarité TF-IDF cosinus `sklearn` — pas de substitution dégradée ;
- le dataset **Brain Language Metrics** (métrique propriétaire, multifactorielle) reste **RECOVERABLE-MACHINE** : il vit sur QC Cloud et alimente le `main.py` compagnon ;
- distinction tenue sans extrapolation : **calcul fini Python** (ce notebook) / **mesure de backtest** (Cloud) / **preuve académique** (*Lazy Prices*, JF 2020) ;
- le verdict de l'article est marginal par ses propres chiffres — ce compagnon sert à *comprendre et vérifier* le signal, pas à le sacraliser.